# Exploring the API

**Goal:** find what every endpoint returns (its schema) and how the endpoints connect.

**Method:** start knowing only the help page and the calendar. Call every endpoint whose parameters we have
values for, keep any values that fill other endpoints' parameters, and repeat until nothing new can be called.

In [31]:
import json
import time
import urllib.parse
import urllib.request
from pathlib import Path

import pandas
from genson import SchemaBuilder

pandas.set_option("display.max_colwidth", None)

API_BASE = "https://api.erg.ic.ac.uk/AirQuality"
CACHE = Path("api_cache")                          # every reply is saved here, so nothing is requested twice
CACHE.mkdir(exist_ok=True)

## 1. Every endpoint, and the parameters it needs

In [32]:
Endpoints = pandas.read_html(API_BASE + "/help")[0]                      # the one table on the help page
Endpoints = Endpoints[Endpoints["Uri"].str.endswith("/Json")].copy()     # keep the JSON versions
Endpoints["Params"] = Endpoints["Uri"].str.findall(r"\{(\w+)\}")        # the names inside {...}
AllParams = set(Endpoints["Params"].sum())                               # every parameter name, once each

print(len(Endpoints), "endpoints,", len(AllParams), "parameters:", sorted(AllParams))
Endpoints[["Uri", "Params"]]

36 endpoints, 25 parameters: ['AirQualityIndex', 'Date', 'EndDate', 'GroupName', 'LocalAuthorityId', 'Period', 'SiteCode', 'SpeciesCode', 'SpeciesName', 'StartDate', 'Step', 'Units', 'Year', 'backA', 'backB', 'backC', 'backD', 'code', 'limit', 'mean', 'montype', 'site', 'skip', 'species', 'year']


,Uri,Params
1,/Annual/MonitoringObjective/GroupName={GroupName}/Json,[GroupName]
3,/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json,"[GroupName, Year]"
5,/Annual/MonitoringObjective/LocalAuthorityId={LocalAuthorityId}/Year={Year}/Json,"[LocalAuthorityId, Year]"
7,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate}/Json,"[SiteCode, StartDate]"
9,/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year}/Json,"[SiteCode, Year]"
11,/Annual/MonitoringReport/SiteCode={SiteCode}/Year={Year}/Json,"[SiteCode, Year]"
12,/Annualiser/Calculate/year={year}/site={site}/backA={backA}/backB={backB}/backC={backC}/backD={backD}/species={species}/mean={mean}/Json,"[year, site, backA, backB, backC, backD, species, mean]"
13,/Annualiser/Sites/year={year}/species={species}/Json,"[year, species]"
15,/Daily/MonitoringIndex/GroupName={GroupName}/Date={Date}/Json,"[GroupName, Date]"
17,/Daily/MonitoringIndex/Latest/GroupName={GroupName}/Json,[GroupName]


## 2. Calling an endpoint

The reply is saved the first time and read from disk after that. A failed call raises an error and saves nothing.

In [33]:
def Call(Path):
    File = CACHE / (Path.strip("/").replace("/", "_").replace("=", "-") + ".json")
    if not File.exists():                                                # only ask the API if we never have
        with urllib.request.urlopen(API_BASE + Path, timeout=60) as Response:
            File.write_bytes(Response.read())
        time.sleep(1)                                                    # be kind to the API
    return File.read_text(encoding="utf-8-sig")                         # the reply, as text

## 3. What we know at the start

Only the calendar: these are the one set of values we choose ourselves. Everything else must be found in a reply.

Three parameters are spelled differently from the fields that supply them. These aliases were written down by
looking at the replies.

In [34]:
Calendar = {"StartDate": ["2026-10-27"],
            "EndDate":   ["2026-10-28"],                                 # EndDate is exclusive: one day of data
            "Date":      ["2026-10-27"],
            "Year":      ["2025"],
            "year":      ["2025"]}
Data = dict(Calendar)                                                    # parameter -> values found so far

Aliases = {"LocalAuthorityCode": "LocalAuthorityId",                     # field name -> parameter name
           "SpeciesCode":        "species",
           "SiteCode":           "site"}

## 4. Keeping the values that fill parameters

`json.loads(Text, object_hook=Harvest)` calls `Harvest` on every `{...}` in a reply, however deeply nested.
`Harvest` does two things for the endpoint being called (`Current`):

- records every field name it returns (`Fields`: the schema, as a list of names, used for the map)
- keeps any value whose field name (without the `@`, or through an alias) is a parameter, and notes which
  endpoint supplied it (`Suppliers`). The calendar parameters are left alone, so a reply's own dates cannot
  replace the ones we chose.

In [35]:
Fields, Suppliers = {}, {}                                               # endpoint -> field names; param -> {endpoint: values}

def Harvest(Dict):
    for Key, Value in Dict.items():
        if isinstance(Value, str):
            Fields[Current].add(Key)                                     # a field this endpoint returns
        for Param in (Key.lstrip("@"), Aliases.get(Key.lstrip("@"))):
            if Param in AllParams and Param not in Calendar and isinstance(Value, str) and Value:
                Suppliers.setdefault(Param, {}).setdefault(Current, set()).add(Value)
                Data.setdefault(Param, [])
                if Value not in Data[Param]:
                    Data[Param].append(Value)
    return Dict                                                          # hand the dict back unchanged

## 5. The crawl

Each round calls every endpoint not yet called whose parameters all have values in `Data`.
An endpoint with parameters is called 3 times, with values spread across what we have; genson merges the
replies into one schema. We stop when a round has nothing to call.

In [36]:
Schemas, Status, Level = {}, {}, {}
Round = 0
while True:
    Ready = [Row for Row in Endpoints.itertuples()
             if Row.Uri not in Status and set(Row.Params) <= set(Data)]     # not called yet, and fillable
    if not Ready:
        break
    for Row in Ready:
        Current = Row.Uri                                                # Harvest records under this endpoint
        Fields[Current] = set()
        Level[Current] = Round
        Builder = SchemaBuilder()
        Status[Row.Uri] = []
        for i in range(3):
            Values = {P: urllib.parse.quote(Data[P][i * len(Data[P]) // 3]) for P in Row.Params}   # URL-safe
            try:
                Text = Call(Row.Uri.format(**Values))
            except Exception as Error:
                Status[Row.Uri].append(str(Error))
                continue
            Status[Row.Uri].append("ok")
            Builder.add_object(json.loads(Text, object_hook=Harvest))    # keep values, then note the schema
            if not Row.Params:
                break                                                    # no parameters: one call is enough
        Schemas[Row.Uri] = Builder.to_schema()
    print(f"Round {Round}: called {len(Ready)} endpoints")
    Round += 1

Round 0: called 6 endpoints
Round 1: called 11 endpoints
Round 2: called 14 endpoints


## 6. What we found

**Status of every endpoint.** `ok` on some calls but not others usually means the sampled value had no data
(e.g. a closed site), not that the endpoint is broken. Endpoints missing from this table were never called:
their parameters appear in no reply.

In [37]:
pandas.Series(Status).to_frame("Results")

,Results
/Hourly/Map/Json,[ok]
/Information/AirPollutionGuide/Json,[ok]
/Information/Groups/Json,[ok]
/Information/IndexHealthAdvice/Json,"[HTTP Error 400: Bad Request, HTTP Error 400: Bad Request, HTTP Error 400: Bad Request]"
/Information/Objective/Json,"[HTTP Error 400: Bad Request, HTTP Error 400: Bad Request, HTTP Error 400: Bad Request]"
/Information/Species/Json,[ok]
/Annual/MonitoringObjective/GroupName={GroupName}/Json,"[ok, ok, ok]"
/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json,"[ok, HTTP Error 400: Bad Request, HTTP Error 400: Bad Request]"
/Annualiser/Sites/year={year}/species={species}/Json,"[ok, ok, ok]"
/Daily/MonitoringIndex/GroupName={GroupName}/Date={Date}/Json,"[ok, ok, ok]"


In [38]:
print("Never called:")
for Uri in Endpoints["Uri"]:
    if Uri not in Status:
        print(" ", Uri)

Never called:
  /Annualiser/Calculate/year={year}/site={site}/backA={backA}/backB={backB}/backC={backC}/backD={backD}/species={species}/mean={mean}/Json
  /Data/DiffusionTube/code={code}/montype={montype}/Json
  /Data/SiteSpecies/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/StartDate={StartDate}/EndDate={EndDate}/Period={Period}/Units={Units}/Step={Step}/Json
  /Data/SiteSpeciesIndexDays/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/Period={Period}/Json
  /Information/News/Skip={skip}/limit={limit}/Json


**Parameter values found, and where they could come from.**

In [39]:
pandas.DataFrame({"Values found": {P: len(V) for P, V in Data.items()},
                  "Examples":     {P: V[:3] for P, V in Data.items()}})

,Values found,Examples
StartDate,1,[2026-10-27]
EndDate,1,[2026-10-28]
Date,1,[2026-10-27]
Year,1,[2025]
year,1,[2025]
SpeciesCode,7,"[NO2, O3, PM10]"
species,7,"[NO2, O3, PM10]"
SpeciesName,6,"[Nitrogen Dioxide, Ozone, PM10 Particulate]"
GroupName,69,"[All, AroundLondon, CambridgeCity]"
SiteCode,391,"[A30, AD1, AR1]"


**A schema**, for one endpoint. Every endpoint's schema is in `Schemas`.

In [40]:
print(json.dumps(Schemas["/Information/MonitoringSites/GroupName={GroupName}/Json"], indent=2))

{
  "$schema": "http://json-schema.org/schema#",
  "type": "object",
  "properties": {
    "Sites": {
      "type": "object",
      "properties": {
        "Site": {
          "type": "array",
          "items": {
            "type": "object",
            "properties": {
              "@LocalAuthorityCode": {
                "type": "string"
              },
              "@LocalAuthorityName": {
                "type": "string"
              },
              "@SiteCode": {
                "type": "string"
              },
              "@SiteName": {
                "type": "string"
              },
              "@SiteType": {
                "type": "string"
              },
              "@DateClosed": {
                "type": "string"
              },
              "@DateOpened": {
                "type": "string"
              },
              "@Latitude": {
                "type": "string"
              },
              "@Longitude": {
                "type": "string"
         

## 7. The map, generated from what this notebook found

- **Columns** are the rounds of the crawl: what could be called once the previous round's values were known.
- **Each endpoint** points to a box of the fields it returns (its schema).
- **Each parameter** is a circle. It is fed by the endpoint that supplied the most values for it (the catalogue),
  and points to every endpoint that needs it.
- **Grey** endpoints never returned data: broken, or never called because no reply supplies their parameters.
- Dates and years are left off: we chose them ourselves (section 3), so no endpoint supplies them.

The diagram is saved as `api_map/map.mmd` (Mermaid text) and `api_map/map.html` (open it in a browser).

In [41]:
from IPython.display import HTML

Main = {P: max(S, key=lambda Uri: len(S[Uri])) for P, S in Suppliers.items()}   # param -> its catalogue endpoint
Id = {Uri: f"E{Number}" for Number, Uri in enumerate(Endpoints["Uri"])}

def Name(Uri):
    return Uri.removeprefix("/").removesuffix("/Json")

Lines = ["flowchart-elk LR", "  classDef nodata fill:#eeeeee,stroke:#999999,color:#666666"]
for Round in sorted(set(Level.values())):                                  # one column per round
    Lines.append(f'  subgraph R{Round}["Round {Round}"]')
    for Uri in [U for U, R in Level.items() if R == Round]:
        Lines.append(f'    {Id[Uri]}["{Name(Uri)}"]')
        if Fields[Uri]:
            Names = sorted(Fields[Uri])
            Rows = [" | ".join(Names[i:i + 4]) for i in range(0, len(Names), 4)]   # 4 fields per line
            Lines.append(f'    {Id[Uri]} --> {Id[Uri]}F["{"<br/>".join(Rows)}"]')
    Lines.append("  end")

Lines.append('  subgraph Never["Never called"]')
for Uri in Endpoints["Uri"]:
    if Uri not in Status:
        Lines.append(f'    {Id[Uri]}["{Name(Uri)}"]')
Lines.append("  end")

for Param, Uri in Main.items():                                            # catalogue -> parameter -> users
    Users = [Id[Row.Uri] for Row in Endpoints.itertuples() if Param in Row.Params]
    Lines.append(f"  {Id[Uri]}F --> P_{Param}(({Param}))")
    Lines.append(f"  P_{Param} --> " + " & ".join(Users))

NoData = [Id[Uri] for Uri in Endpoints["Uri"] if "ok" not in Status.get(Uri, [])]
Lines.append("  class " + ",".join(NoData) + " nodata")

Map = "\n".join(Lines)
Page = f"""<pre class="mermaid">{Map}</pre>
<script src="https://cdn.jsdelivr.net/npm/mermaid@10.9.1/dist/mermaid.min.js"></script>
<script>mermaid.initialize({{startOnLoad: true}});</script>"""
Path("api_map").mkdir(exist_ok=True)
Path("api_map/map.mmd").write_text(Map)
Path("api_map/map.html").write_text(f"<!doctype html><html><body>{Page}</body></html>")
HTML(Page)